In [ ]:
from cross_talk_lib import *

# Set parameters
CLASSIC_IN = 1
CLASSIC_OUT = 65
QUANTUM_OUT = 10
QUANTUM_INPUTS = range(1, 65)
SWITCH_API = "http://192.168.56.10:8008/api/data/optical-switch:cross-connects"
BROKER_URL = "http://localhost:5672/"
CAPABILITY_ID = "607f45911560b7454c8365dc705cc2f7725adfcdf284f65ff8a7f9f060b55ea6"
CSV_FILE = "data/data-clink-fixed-qlink-var-new-laser.csv"

# Init
mp_client = init_mp_client(BROKER_URL)

# Cleanup any existing cross-connects
delete_all_crossconnects(SWITCH_API)

# Create Classical cross-connect
create_crossconnect(CLASSIC_IN, CLASSIC_OUT, SWITCH_API)


True

In [ ]:
avr_c = measure_counts(
    mp_client=mp_client,
    capability_id=CAPABILITY_ID,
    channel=1,
    timeout=5
)

In [ ]:
# Run for each quantum port
import pandas as pd
data = []

for qin in QUANTUM_INPUTS:
    if qin == CLASSIC_IN:
        continue

    ok = create_crossconnect(qin, QUANTUM_OUT, SWITCH_API)
    if not ok:
        print(f"Failed to connect {qin} -> {QUANTUM_OUT}")
        continue

    timestamp = time.time()
    utc_time = datetime.utcnow().isoformat()
    start = time.time()
    count = measure_counts(mp_client, CAPABILITY_ID, channel=1)
    end = time.time()
    duration = end - start

    row = {
        "CLASSIC_IN": CLASSIC_IN,
        "CLASSIC_OUT": CLASSIC_OUT,
        "QUANT_IN": qin,
        "QUANT_OUT": QUANTUM_OUT,
        "Count": count,
        "Timestamp": timestamp,
        "UTC": utc_time,
        "Duration": duration
    }

    print(row)
    data.append(row)

# Save to CSV
df = pd.DataFrame(data)
df.to_csv(CSV_FILE, mode='a', index=False)

# Final cleanup
delete_all_crossconnects(SWITCH_API)
